# SmolVLA ファインチューニング：牡蠣人形の言語条件付け（箱/お辞儀/赤袋）

**目的**：同じ牡蠣＋赤袋のシーンで、**言語指示だけ**で行動が変わるVLAを学習する。
- `pick up the white oyster plush and put it in the box`（箱, 20本）
- `pick up the white oyster plush and make it bow`（お辞儀, 20本）
- `pick up the red snack bag and put it in the box`（赤袋→箱, 20本）
計60エピソード。

**事前準備**
1. `so101_oyster_vla_dataset.zip` を Googleドライブ `MyDrive/so101/` にアップロード
2. 短時間ラン=T4でOK（疎通確認）／長時間ラン=**A100に切替**推奨

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ZIP = "/content/drive/MyDrive/so101/so101_oyster_vla_dataset.zip"

!mkdir -p /content/dataset
!unzip -q "$DRIVE_ZIP" -d /content/dataset
!ls /content/dataset

In [ ]:
# lerobot + PEFT(LoRA) + SmolVLAプロセッサ用num2words。torchaoはPEFT要求(>=0.16)へ明示アップグレード
!pip install -q "lerobot[dataset,training]==0.6.0" accelerate peft num2words "torchao>=0.16.0"

In [ ]:
import glob

candidates = glob.glob("/content/dataset/so101_oyster_vla_*")
assert len(candidates) == 1, f"データセットフォルダが1つに絞れません: {candidates}"
DATASET_ROOT = candidates[0]
print("DATASET_ROOT =", DATASET_ROOT)

EPISODES = "[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59]"  # 全60エピソード

## 短時間ラン（T4でOK・疎通確認、目安5000ステップ）

In [ ]:
!lerobot-train \
  --dataset.repo_id=local/so101_oyster_vla \
  --dataset.root="$DATASET_ROOT" \
  --dataset.episodes="$EPISODES" \
  --dataset.video_backend=pyav \
  --policy.type=smolvla \
  --policy.pretrained_path=lerobot/smolvla_base \
  --policy.load_vlm_weights=true \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --peft.r=16 \
  --output_dir=/content/drive/MyDrive/so101/train_output/smolvla_oyster_short \
  --job_name=smolvla_oyster_short \
  --steps=5000 \
  --batch_size=8 \
  --save_freq=1000 \
  --log_freq=50 \
  --wandb.enable=false

## 長時間ラン（★A100に切替してから実行推奨、目安50000ステップ）

「ランタイム→ランタイムのタイプを変更」でA100へ。切替でセッションがリセットされるので上のセルから順に再実行。

In [ ]:
!lerobot-train \
  --dataset.repo_id=local/so101_oyster_vla \
  --dataset.root="$DATASET_ROOT" \
  --dataset.episodes="$EPISODES" \
  --dataset.video_backend=pyav \
  --policy.type=smolvla \
  --policy.pretrained_path=lerobot/smolvla_base \
  --policy.load_vlm_weights=true \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --peft.r=16 \
  --output_dir=/content/drive/MyDrive/so101/train_output/smolvla_oyster_long \
  --job_name=smolvla_oyster_long \
  --steps=50000 \
  --batch_size=8 \
  --save_freq=5000 \
  --log_freq=100 \
  --wandb.enable=false

In [ ]:
!echo "--- short ---"; ls -la /content/drive/MyDrive/so101/train_output/smolvla_oyster_short/checkpoints
!echo "--- long ---"; ls -la /content/drive/MyDrive/so101/train_output/smolvla_oyster_long/checkpoints